<a href="https://colab.research.google.com/github/manaskanugo97-max/Healthcare-Lead-Gen-System/blob/main/scorer_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
import os

print("Project files:")
print(os.listdir())

print("\nData files:")
print(os.listdir("data"))

Project files:
['data', '__pycache__', 'osm_scraper.py', 'online_presence_checker.py', 'config.py', 'website_analyzer.py', 'scorer.py']

Data files:
['healthcare_osm_raw.csv', 'healthcare_online_presence.csv', 'healthcare_website_analysis.csv', 'healthcare_scored_leads.csv']


In [ ]:
%%writefile config.py

import os

PROJECT_DIR = "/content/drive/MyDrive/healthcare_lead_gen_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

# Step 1 output
RAW_OSM_CSV = os.path.join(DATA_DIR, "healthcare_osm_raw.csv")

# Step 2 output
ONLINE_PRESENCE_CSV = os.path.join(DATA_DIR, "healthcare_online_presence.csv")

# Step 3 output
WEBSITE_ANALYSIS_CSV = os.path.join(DATA_DIR, "healthcare_website_analysis.csv")

# Step 4 output
SCORED_LEADS_CSV = os.path.join(DATA_DIR, "healthcare_scored_leads.csv")


FINAL_CSV = os.path.join(DATA_DIR, "healthcare_leads_final.csv")

Overwriting config.py


In [ ]:
from config import WEBSITE_ANALYSIS_CSV, SCORED_LEADS_CSV
import os

print("Step 4 input file:")
print(WEBSITE_ANALYSIS_CSV)

print("\nDoes Step 4 input file exist?")
print(os.path.exists(WEBSITE_ANALYSIS_CSV))

print("\nStep 4 output file will be:")
print(SCORED_LEADS_CSV)

Step 4 input file:
/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_website_analysis.csv

Does Step 4 input file exist?
True

Step 4 output file will be:
/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_scored_leads.csv


In [ ]:
%%writefile scorer.py

import os
import pandas as pd

from config import WEBSITE_ANALYSIS_CSV, SCORED_LEADS_CSV


def clean_value(value):
    """
    Convert NaN/empty values into clean string.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


def calculate_lead_score(row):
    """
    Calculates lead score based on online presence and digital weakness.

    Higher score means better outreach opportunity.
    """

    score = 0
    reasons = []

    website_classification = clean_value(row.get("Website Classification", ""))
    online_presence_type = clean_value(row.get("Online Presence Type", ""))
    official_website = clean_value(row.get("Official Website Found", ""))
    phone = clean_value(row.get("Phone Number", ""))
    email = clean_value(row.get("Email Address", ""))
    referral_links = clean_value(row.get("Referral Platform Links", ""))
    contact_page = clean_value(row.get("Contact Page Found", ""))
    services_page = clean_value(row.get("Services Page Found", ""))
    https_used = clean_value(row.get("HTTPS Used", ""))

    # Website-based scoring
    if website_classification == "No Website":
        score += 35
        reasons.append("No official website found")
    elif website_classification == "Poor Website":
        score += 25
        reasons.append("Website exists but digital quality is weak")
    elif website_classification == "Good Website":
        score += 10
        reasons.append("Business already has a good website")

    # Online presence scoring
    if online_presence_type in ["Referral Platform Only", "Referral Platform"]:
        score += 20
        reasons.append("Business depends mainly on referral/listing platforms")
    elif online_presence_type == "No Strong Online Presence Found":
        score += 25
        reasons.append("Business has weak online visibility")
    elif online_presence_type in ["Official Website Found", "Official Website"]:
        score += 5
        reasons.append("Official website presence found")

    # Contact availability
    if phone:
        score += 15
        reasons.append("Phone number is available for outreach")
    else:
        score -= 5
        reasons.append("Phone number not available")

    if email:
        score += 10
        reasons.append("Email address is available")
    else:
        reasons.append("Email address not available")

    # Website weakness details
    if official_website:
        if https_used == "No":
            score += 5
            reasons.append("Website does not clearly use HTTPS")

        if contact_page == "No":
            score += 5
            reasons.append("Contact or appointment page not clearly found")

        if services_page == "No":
            score += 5
            reasons.append("Services or treatment page not clearly found")

    # Referral platform indicates business exists online but may lack direct brand control
    if referral_links:
        score += 5
        reasons.append("Business appears on third-party platforms")

    # Keep score between 0 and 100
    score = max(0, min(score, 100))

    return score, "; ".join(reasons)


def classify_potential(score):
    """
    Converts score into High / Medium / Low potential.
    """

    if score >= 60:
        return "High"
    elif score >= 35:
        return "Medium"
    else:
        return "Low"


def generate_simple_reason(row, score, potential):
    """
    Creates clean final business potential reason.
    """

    website_classification = clean_value(row.get("Website Classification", ""))
    online_presence_type = clean_value(row.get("Online Presence Type", ""))
    phone = clean_value(row.get("Phone Number", ""))

    if potential == "High":
        if website_classification == "No Website":
            return "Business has local presence but no official website, making it a strong outreach opportunity."
        elif website_classification == "Poor Website":
            return "Business has an online presence but lacks a strong modern digital platform."
        elif "Referral" in online_presence_type:
            return "Business appears to rely mainly on third-party listing platforms instead of its own strong digital presence."
        else:
            return "Business shows outreach potential due to gaps in digital visibility."

    elif potential == "Medium":
        if phone:
            return "Business has some public contact information and moderate opportunity for digital improvement."
        else:
            return "Business has some online presence but limited contact or digital information."

    else:
        return "Business already has stronger digital presence or limited available outreach information."


def score_leads(input_file=WEBSITE_ANALYSIS_CSV, output_file=SCORED_LEADS_CSV):
    """
    Reads Step 3 file and creates Step 4 scored lead file.
    """

    print("Step 4 started")
    print("Input file:", input_file)
    print("Output file:", output_file)

    if not os.path.exists(input_file):
        raise FileNotFoundError(
            f"Input file not found: {input_file}. Please complete Step 3 first."
        )

    df = pd.read_csv(input_file)

    lead_scores = []
    potential_categories = []
    detailed_reasons = []
    final_reasons = []

    for index, row in df.iterrows():
        score, detailed_reason = calculate_lead_score(row)
        potential = classify_potential(score)
        final_reason = generate_simple_reason(row, score, potential)

        lead_scores.append(score)
        potential_categories.append(potential)
        detailed_reasons.append(detailed_reason)
        final_reasons.append(final_reason)

    df["Lead Score"] = lead_scores
    df["Business Potential Category"] = potential_categories
    df["Detailed Score Reason"] = detailed_reasons
    df["Reason for Classification"] = final_reasons

    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Step 4 completed")
    print("Total scored leads:", len(df))
    print("Saved file:", output_file)

    return df


if __name__ == "__main__":
    df = score_leads(
        input_file=WEBSITE_ANALYSIS_CSV,
        output_file=SCORED_LEADS_CSV
    )

    print(df[[
        "Business Name",
        "Website Classification",
        "Online Presence Type",
        "Lead Score",
        "Business Potential Category",
        "Reason for Classification"
    ]].head())

Overwriting scorer.py


In [ ]:
!python scorer.py

Step 4 started
Input file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_website_analysis.csv
Output file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_scored_leads.csv
Step 4 completed
Total scored leads: 100
Saved file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_scored_leads.csv
                                       Business Name  ...                          Reason for Classification
0                           Greater Kailash Hospital  ...  Business already has stronger digital presence...
1                                      Qurban Husain  ...  Business already has stronger digital presence...
2  Shri Aurobindo dental clinic and institute of ...  ...  Business already has stronger digital presence...
3                                          Dr. Mutha  ...  Business has some public contact information a...
4                                        Arshid shah  ...  Business has some public contact information a.

In [ ]:
import pandas as pd
import os
from config import SCORED_LEADS_CSV

print("Step 4 file exists:", os.path.exists(SCORED_LEADS_CSV))
print("Step 4 file path:", SCORED_LEADS_CSV)

df4 = pd.read_csv(SCORED_LEADS_CSV)

print("Total rows:", len(df4))
print("Columns:")
print(df4.columns.tolist())

df4[[
    "Business Name",
    "Website Classification",
    "Online Presence Type",
    "Lead Score",
    "Business Potential Category",
    "Reason for Classification"
]].head()

Step 4 file exists: True
Step 4 file path: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_scored_leads.csv
Total rows: 100
Columns:
['Unnamed: 0', 'Business Name', 'Industry Category', 'Business Description', 'Location', 'Latitude', 'Longitude', 'Google Maps Profile Link', 'OpenStreetMap Link', 'Website URL', 'Phone Number', 'Email Address', 'Source', 'Official Website Found', 'Online Presence Type', 'Referral Platform Links', 'Search Result Links', 'Website Classification', 'Lead Score', 'Business Potential Category', 'Detailed Score Reason', 'Reason for Classification']


,Business Name,Website Classification,Online Presence Type,Lead Score,Business Potential Category,Reason for Classification
0,Greater Kailash Hospital,Poor Website,Official Website Found,30,Low,Business already has stronger digital presence...
1,Qurban Husain,Poor Website,Official Website Found,30,Low,Business already has stronger digital presence...
2,Shri Aurobindo dental clinic and institute of ...,Poor Website,Official Website Found,30,Low,Business already has stronger digital presence...
3,Dr. Mutha,Poor Website,Official Website Found,50,Medium,Business has some public contact information a...
4,Arshid shah,Poor Website,Official Website Found,55,Medium,Business has some public contact information a...
